
# Phase 7 — SHML System Integration

## Objective

Integrate the existing SHML components into a
reproducible end-to-end machine-learning workflow.

## Step 7.1 — Safety Controller Verification

Verify that the original Phase 5 safety controller
correctly accepts and rejects recovery candidates.

## Scope

- Dataset: UCI Default of Credit Card Clients
- Model: Logistic Regression
- Failure type: BILL_AMT1 missingness
- Recovery: Training-median imputation
- Safety: F1, recall, and data-integrity checks


In [11]:

# Phase 7.1 — Test original safety controller

import numpy as np
import pandas as pd

print("RUNNING PHASE 7.1 SAFETY TESTS")

healthy_metrics = {
    "accuracy": 0.8087,
    "precision": 0.6948,
    "recall": 0.2402,
    "f1": 0.3570,
    "roc_auc": 0.7130
}

test_cases = [
    ("Excessive F1 loss", 0.75, 1.00, True, "REJECT"),
    ("Excessive recall loss", 1.00, 0.75, True, "REJECT"),
    ("Data integrity failure", 1.00, 1.00, False, "REJECT"),
    ("Valid recovery", 0.95, 0.95, True, "ACCEPT")
]

results = []

for name, f1_factor, recall_factor, integrity, expected in test_cases:
    candidate = healthy_metrics.copy()
    candidate["f1"] *= f1_factor
    candidate["recall"] *= recall_factor

    outcome = evaluate_recovery_safety(
        recovered_metrics=candidate,
        healthy_metrics=healthy_metrics,
        data_integrity_passed=integrity
    )

    results.append({
        "Test": name,
        "Expected": expected,
        "Actual": outcome["decision"],
        "Passed": outcome["decision"] == expected
    })

phase7_safety_results = pd.DataFrame(results)

print(phase7_safety_results.to_string(index=False))

assert phase7_safety_results["Passed"].all()

print("\nPhase 7.1 verification passed.")


RUNNING PHASE 7.1 SAFETY TESTS
                  Test Expected Actual  Passed
     Excessive F1 loss   REJECT REJECT    True
 Excessive recall loss   REJECT REJECT    True
Data integrity failure   REJECT REJECT    True
        Valid recovery   ACCEPT ACCEPT    True

Phase 7.1 verification passed.


In [12]:

# src/safety.py
# SHML recovery safety controller

import numpy as np


def evaluate_recovery_safety(
    recovered_metrics,
    healthy_metrics,
    data_integrity_passed,
    max_f1_loss=0.10,
    max_recall_loss=0.10
):
    """
    Evaluate offline recovery acceptance criteria.

    Returns:
        dict: Safety decision, reason, and performance losses.
    """

    required_metrics = [
        "accuracy", "precision", "recall", "f1", "roc_auc"
    ]

    metrics_valid = all(
        metric in recovered_metrics
        and np.isfinite(recovered_metrics[metric])
        and 0 <= recovered_metrics[metric] <= 1
        for metric in required_metrics
    )

    baseline_valid = (
        healthy_metrics["f1"] > 0
        and healthy_metrics["recall"] > 0
    )

    if not metrics_valid or not baseline_valid:
        return {
            "decision": "REJECT",
            "reason": "Invalid evaluation metrics or baseline",
            "f1_loss_pct": np.nan,
            "recall_loss_pct": np.nan
        }

    f1_loss = (
        healthy_metrics["f1"] - recovered_metrics["f1"]
    ) / healthy_metrics["f1"]

    recall_loss = (
        healthy_metrics["recall"] - recovered_metrics["recall"]
    ) / healthy_metrics["recall"]

    reasons = []

    if not data_integrity_passed:
        reasons.append("Data integrity check failed")

    if f1_loss > max_f1_loss:
        reasons.append("F1 loss exceeds tolerance")

    if recall_loss > max_recall_loss:
        reasons.append("Recall loss exceeds tolerance")

    decision = "REJECT" if reasons else "ACCEPT"

    return {
        "decision": decision,
        "reason": "; ".join(reasons) if reasons else "All checks passed",
        "f1_loss_pct": f1_loss * 100,
        "recall_loss_pct": recall_loss * 100
    }


In [14]:

# Phase 7.2 — Verify reusable safety module
from src.safety import evaluate_recovery_safety as module_safety

test_result = module_safety(
    recovered_metrics={
        "accuracy": 0.8087,
        "precision": 0.6948,
        "recall": 0.2402,
        "f1": 0.3570,
        "roc_auc": 0.7130
    },
    healthy_metrics={
        "f1": 0.3570,
        "recall": 0.2402
    },
    data_integrity_passed=True
)

print("Decision:", test_result["decision"])
print("Reason:", test_result["reason"])

assert test_result["decision"] == "ACCEPT"

print("Phase 7.2 safety module import passed.")


Decision: ACCEPT
Reason: All checks passed
Phase 7.2 safety module import passed.


In [17]:

# Phase 7.2 — Verify detection module

import pandas as pd
from src.detection import detect_feature_missingness

test_batch = pd.DataFrame({
    "BILL_AMT1": [100, 200, None, None, 500]
})

result = detect_feature_missingness(
    X_batch=test_batch,
    feature="BILL_AMT1",
    reference_missingness=0.0,
    threshold=0.05
)

print("Current missingness:", result["current_missingness"])
print("Missingness increase:", result["missingness_increase"])
print("Failure detected:", result["failure_detected"])

assert result["failure_detected"] is True

print("Phase 7.2 detection module verification passed.")


Current missingness: 0.4
Missingness increase: 0.4
Failure detected: True
Phase 7.2 detection module verification passed.


In [19]:


# Phase 7.2 — Verify diagnosis module
from src.diagnosis import diagnose_failure

test_detection = {
    "feature": "BILL_AMT1",
    "current_missingness": 0.15,
    "failure_detected": True
}

diagnosis_result = diagnose_failure(test_detection)

print("Feature:", diagnosis_result["feature"])
print("Failure type:", diagnosis_result["failure_type"])
print("Severity:", diagnosis_result["severity"])
print("Recovery action:", diagnosis_result["recovery_action"])
print("Explanation:", diagnosis_result["explanation"])

assert diagnosis_result["severity"] == "Moderate"
assert diagnosis_result["recovery_action"] == "Median imputation"

print("\nPhase 7.2 diagnosis module verification passed.")


Feature: BILL_AMT1
Failure type: Numerical missingness
Severity: Moderate
Recovery action: Median imputation
Explanation: BILL_AMT1 has 15.00% missing values. Severity: Moderate. Recommended action: Median imputation.

Phase 7.2 diagnosis module verification passed.


In [21]:

# Phase 7.2 — Verify recovery module
from src.recovery import recover_missingness

test_batch = pd.DataFrame({
    "BILL_AMT1": [1000, None, 3000, None, 5000],
    "AGE": [25, 30, 35, 40, 45]
})

test_diagnosis = {
    "feature": "BILL_AMT1",
    "severity": "Moderate",
    "recovery_action": "Median imputation"
}

# Training median from the original SHML experiment
training_median = 22225.5

recovered_batch, recovery_log = recover_missingness(
    X_batch=test_batch,
    diagnosis=test_diagnosis,
    training_median=training_median
)

print("Original BILL_AMT1:")
print(test_batch["BILL_AMT1"].tolist())

print("\nRecovered BILL_AMT1:")
print(recovered_batch["BILL_AMT1"].tolist())

print("\nRecovery log:")
print(recovery_log)

assert recovery_log["missing_before"] == 2
assert recovery_log["missing_after"] == 0
assert recovery_log["values_imputed"] == 2

# Verify that original data was not modified
assert test_batch["BILL_AMT1"].isna().sum() == 2

# Verify unrelated feature remains unchanged
assert recovered_batch["AGE"].equals(test_batch["AGE"])

print("\nPhase 7.2 recovery module verification passed.")


Original BILL_AMT1:
[1000.0, nan, 3000.0, nan, 5000.0]

Recovered BILL_AMT1:
[1000.0, 22225.5, 3000.0, 22225.5, 5000.0]

Recovery log:
{'feature': 'BILL_AMT1', 'severity': 'Moderate', 'action': 'Median imputation', 'missing_before': 2, 'missing_after': 0, 'values_imputed': 2, 'recovery_applied': True}

Phase 7.2 recovery module verification passed.


In [22]:

# Phase 7.3 — Import all SHML components

from src.detection import detect_feature_missingness
from src.diagnosis import diagnose_failure
from src.recovery import recover_missingness
from src.safety import evaluate_recovery_safety

print("All four SHML modules imported successfully.")


All four SHML modules imported successfully.


In [23]:

# Phase 7.3.2 — Integrate detection, diagnosis, recovery

import pandas as pd

# Small test batch with 20% missingness
X_test_batch = pd.DataFrame({
    "BILL_AMT1": [1000, 2000, None, 4000, 5000],
    "AGE": [25, 30, 35, 40, 45]
})

# 1. Detect
detection_result = detect_feature_missingness(
    X_batch=X_test_batch,
    feature="BILL_AMT1",
    reference_missingness=0.0,
    threshold=0.05
)

# 2. Diagnose
diagnosis_result = diagnose_failure(detection_result)

# 3. Recover
X_recovered, recovery_log = recover_missingness(
    X_batch=X_test_batch,
    diagnosis=diagnosis_result,
    training_median=22225.5
)

print("Detection:", detection_result["failure_detected"])
print("Severity:", diagnosis_result["severity"])
print("Recovery action:", diagnosis_result["recovery_action"])
print("Missing before:", recovery_log["missing_before"])
print("Missing after:", recovery_log["missing_after"])

assert detection_result["failure_detected"] is True
assert diagnosis_result["severity"] == "Moderate"
assert recovery_log["missing_after"] == 0
assert X_test_batch["BILL_AMT1"].isna().sum() == 1

print("\nPhase 7.3.2 integration test passed.")


Detection: True
Severity: Moderate
Recovery action: Median imputation
Missing before: 1
Missing after: 0

Phase 7.3.2 integration test passed.
